## Paso 1: Preparar el modelo de embeddings

In [0]:
%pip install mlflow torch transformers pandas==2.2.2 numpy==1.26.4
%restart_python

## Paso 2A: Crear una función para generar embeddings

In [0]:
import mlflow
import numpy as np
import pandas as pd
import torch
from typing import List
from transformers import AutoTokenizer, AutoModel

class SimpleEmbeddingModel(mlflow.pyfunc.PythonModel):
    """
    Un modelo de embedding personalizado para MLflow que es compatible
    con Databricks Vector Search.
    
    Esta clase carga un modelo de 'transformers', define la lógica para
    generar embeddings y formatea la salida específicamente como
    lo requiere Databricks Vector Search.
    """
    
    def load_context(self, context):
        """
        Esta función se ejecuta una sola vez cuando el modelo se carga en memoria
        en el endpoint de Model Serving. Es el lugar ideal para cargar modelos pesados.
        """
        # Carga el tokenizador y el modelo pre-entrenado desde Hugging Face
        self.tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
        self.model = AutoModel.from_pretrained("distilbert-base-uncased")
    
    def encode(self, texts: List[str]) -> np.ndarray:
        """
        Función auxiliar para procesar una lista de textos y devolver
        sus embeddings como un array de NumPy.
        """
        # Tokeniza el lote de textos. padding y truncation aseguran que todas
        # las secuencias tengan la misma longitud.
        encoded_input = self.tokenizer(
            texts, padding=True, truncation=True, return_tensors='pt'
        )
        
        # Infiere con el modelo sin calcular gradientes para ahorrar memoria y tiempo.
        with torch.no_grad():
            model_output = self.model(**encoded_input)
            
        # Calcula los embeddings. Usamos la media de los 'last_hidden_state'
        # como una estrategia de pooling simple y efectiva.
        embeddings = model_output.last_hidden_state.mean(dim=1)
        
        # Devuelve los embeddings como un array de NumPy.
        return embeddings.numpy()

    def predict(self, context, model_input: pd.DataFrame) -> List[List[float]]:
        """
        El método principal que MLflow llama para cada petición de inferencia.
        Toma un DataFrame, extrae el texto, genera los embeddings y devuelve
        la salida en el formato requerido por Databricks Vector Search.
        """
        # --- 1. Validación y Procesamiento de la Entrada ---
        # Asegura que la entrada sea un DataFrame de Pandas.
        if not isinstance(model_input, pd.DataFrame):
            raise TypeError(f"La entrada debe ser un DataFrame de pandas, pero se recibió {type(model_input)}")

        # Verifica que la columna 'input' requerida exista.
        if "input" not in model_input.columns:
            raise ValueError("El DataFrame de entrada debe contener una columna llamada 'input'")

        # Extrae la columna de texto, rellena valores nulos con un string vacío
        # para evitar errores, convierte todo a string y luego a una lista de Python.
        texts = model_input["input"].fillna('').astype(str).tolist()

        # --- 2. Generación de Embeddings ---
        # Llama a la función de codificación y se asegura de que el tipo sea float32.
        embeddings = self.encode(texts).astype(np.float32)

        # --- 3. Formato de Salida para Vector Search ---
        # Devuelve los embeddings como una lista de listas de floats.
        # Este es el formato exacto y único que Databricks Vector Search espera.
        # El wrapper de MLflow pyfunc lo convertirá en: {"predictions": [[...], [...]]}
        return embeddings.tolist()

## Paso 3A: Registrar el modelo en MLflow (Databricks)

In [0]:
import mlflow
import mlflow.pyfunc  # <--- CORREGIDO
import torch
import transformers
import pandas as pd
from mlflow.models.signature import infer_signature

# (Asumo que la clase SimpleEmbeddingModel está definida previamente en el mismo script/notebook)

# Set registry URI
mlflow.set_registry_uri("databricks-uc")
model_name = "bluetab.rag.simple_embedding_model_bluetab"

# Ejemplo de input
input_example = pd.DataFrame({"input": ["Your string for the embedding model goes here"]})

# Instanciamos el modelo y generamos output para la firma
# Esto asegura que la firma de salida sea correcta
model_predict = SimpleEmbeddingModel()
model_predict.load_context(None)
output_example = model_predict.predict(None, input_example)

# Inferencia de la firma
signature = infer_signature(input_example, output_example)
print("Signature inferred successfully:")
print(signature)

# Registro del modelo
with mlflow.start_run(run_name="Register Simple Embedding Model"):
    mlflow.pyfunc.log_model(  # <--- CORREGIDO
        artifact_path="simple_embedding_model", # Nombre del artefacto en la ejecución de MLflow
        python_model=SimpleEmbeddingModel(),
        registered_model_name=model_name,
        input_example=input_example,
        signature=signature,
        pip_requirements=[
            f"mlflow=={mlflow.__version__}",
            f"torch==2.7.1", # <--- SUGERENCIA: Versión dinámica
            f"transformers=={transformers.__version__}",
            "pandas==2.2.2",  # <-- Fijar versión de pandas
            "numpy==1.26.4"   # <-- Fijar versión de numpy compatible
        ]
    )

print(f"Model '{model_name}' registered successfully.")

In [0]:
output_example

## Paso 2B: Crear una función para generar embeddings

In [0]:
%pip install --upgrade sentence-transformers==4.1.0 huggingface_hub
dbutils.library.restartPython()

In [0]:
import mlflow
import mlflow.pyfunc
import numpy as np
import pandas as pd
from typing import List

class MiniLMEmbeddingModel(mlflow.pyfunc.PythonModel):

    def load_context(self, context):
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer("all-MiniLM-L6-v2")

    def encode(self, texts: List[str]):
        embeddings = self.model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
        return embeddings

    def predict(self, context, model_input):
        if not isinstance(model_input, pd.DataFrame):
            raise TypeError(f"Input must be a pandas.DataFrame, but got {type(model_input)}")

        if "input" not in model_input.columns:
            raise ValueError("The input DataFrame must contain a column named 'input'")

        texts = model_input["input"].fillna('').astype(str).tolist()
        embeddings = self.encode(texts).astype(np.float32)

        return [
            {
                "embedding": embedding.tolist(),
                "index": float(idx),
                "vector_dim": float(embeddings.shape[1])
            }
            for idx, embedding in enumerate(embeddings)
        ]

## Paso 3B: Registrar el modelo en MLflow (Databricks)

In [0]:
import mlflow
from mlflow.models.signature import infer_signature

mlflow.set_registry_uri("databricks-uc")
registered_model_name = "bluetab.rag.all_minilm_embedding_model"

# Input de ejemplo
input_example = {
  "input": "Your string for the embedding model goes here"
}
# Generar output para la firma
model_predict = MiniLMEmbeddingModel()
model_predict.load_context(None)  # porque no estás en MLflow
output_example = model_predict.predict(None, input_example)

# Inferir la firma
signature = infer_signature(input_example, output_example)

# Registrar el modelo con input_example y signature
with mlflow.start_run(run_name="Register Model") as run:
    mlflow.pyfunc.log_model(
        name="complex_model",
        python_model= MiniLMEmbeddingModel(),
        registered_model_name=registered_model_name,
        input_example=input_example,
        pip_requirements=[
                "mlflow==" + mlflow.__version__,
                "torch==2.7.1",
            ]
    )

In [0]:
output_example